## Querying the Knowledge Graph with Natural Language

In this demo, we’ll query a knowledge graph using natural language questions.

The goal is to show how a question is:
- Translated into a graph query
- Executed against connected entities
- Returned as structured evidence
- Explained back in natural language

In [7]:

from neo4j import GraphDatabase

# 🔐 Replace with your own connection details
NEO4J_URI = "bolt://localhost:7687"      # Or your Neo4j Aura DB URI
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "neo4j123"              # Replace with your password

# Create the Neo4j driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


## Generating Cypher from Natural Language

At this stage, the LLM is not answering the question.

Its only responsibility is to:
- Read the user question
- Read the graph schema
- Generate a Cypher query that matches the graph structure

This separation is important: query generation happens before execution or explanation.


In [8]:
from langchain_community.graphs import Neo4jGraph
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI



# Connect to graph
graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASSWORD)

cypher_prompt = PromptTemplate.from_template("""
You are a Cypher expert. Given a user question and the full Neo4j schema, write a correct Cypher query using as many MATCH clauses as necessary. 
Do not assume direct relationships unless explicitly defined in the schema. Always follow the correct direction of relationships.
The query needs to return all information about the result.

Schema:
{schema}

Question:
{question}

Cypher Query:
""")





In [9]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def build_chain(graph):
    schema = graph.get_schema
    chain = (
        {
            "question": RunnablePassthrough(),
            "schema": lambda _: schema
        }
        | cypher_prompt
        | llm
        | RunnablePassthrough()
    )
    return chain



## Wrapping Query Generation and Execution

To make graph querying reusable, we wrap the full process in a helper function.

This function:
1. Accepts a natural language question
2. Uses the LLM to generate a Cypher query
3. Executes that query against the graph
4. Returns the raw results

This mirrors how an agent would interact with a graph tool in practice.


In [10]:
generate_cypher = build_chain(graph)

def answer_query(question: str):
    cypher_response = generate_cypher.invoke({"question": question})

    # ✅ Properly extract raw Cypher string
    if isinstance(cypher_response, dict) and "content" in cypher_response:
        cypher_query = cypher_response["content"]
    elif hasattr(cypher_response, 'content'):
        cypher_query = cypher_response.content
    elif isinstance(cypher_response, str):
        # Strip accidental "content=" prefix if present
        cypher_query = cypher_response.strip().removeprefix("content=").strip()
    else:
        raise ValueError("Unexpected Cypher response format.")

    print("📄 LLM Generated Cypher Query:\n", cypher_query)

    # ✅ Run only the extracted Cypher
    result = graph.query(cypher_query)
    print("\n📊 Query Result:\n", result)

    return result



### 🧭 Step: A New Agent Question

We simulate this user query:
> "How has the issue Battery Drain been resolved for Laptop Model X?"

This triggers a multi-hop traversal across the graph to retrieve:
- The issue affecting Laptop Model X  
- Any resolution  
- Supporting evidence from tickets


In [11]:
question = "How has the issue Battery Drain been resolved for Laptop Model X?"
results = answer_query(question)


📄 LLM Generated Cypher Query:
 MATCH (c:Customer)-[:HAS_TICKET]->(t:Ticket)-[:REPORTS]->(i:Issue {name: 'Battery Drain'})-[:AFFECTS]->(p:Product {name: 'Laptop Model X'})
MATCH (t)-[:RESOLVED_BY]->(r:Resolution)
RETURN c, t, i, p, r

📊 Query Result:
 [{'c': {'name': 'Customer 2345', 'customer_id': 2345}, 't': {'ticket_date': '2023-10-08', 'name': 'TKT-001'}, 'i': {'name': 'Battery Drain'}, 'p': {'name': 'Laptop Model X'}, 'r': {'name': 'Patch 1.2.1', 'released_on': '2023-10-10'}}]


### Convert Query Result to Natural Language Answer

Now that we have retrieved the raw Cypher query result from the Neo4j database, we pass it — along with the original question — to a language model to generate a final user-facing answer.

This helps bridge structured data and human-friendly responses.

✅ Input:  
- Question: *"How has the issue Battery Drain been resolved for Laptop Model X?"*  
- Query result: Neo4j structured output

🧠 Output:  
A clear, concise answer generated by the LLM.


In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Prompt for turning structured Cypher result into a natural answer
qa_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Based on the user's question and the Cypher query result from a Neo4j database, provide a clear natural language answer.
Question:
{question}

Cypher Query Result:
{result}

Answer:
""")

final_answer_chain = qa_prompt | llm

# Invoke the chain with the actual question and result
final_response = final_answer_chain.invoke({
    "question": question,
    "result": results
})

print(final_response.content)


The issue of Battery Drain for Laptop Model X has been resolved by releasing Patch 1.2.1 on October 10, 2023. This resolution was associated with Ticket TKT-001 raised by Customer 2345 on October 8, 2023.


## 🔍 Querying Strategy: Demo vs Production

At this point, it’s useful to step back and examine the approach used in this demo.

---

### 🧪 Demo Approach: LLM-Generated Queries

In this notebook, the LLM generates raw graph queries directly from natural language.

✅ **Why this works well**
- Fast to prototype
- Flexible and expressive
- No upfront query definitions required

⚠️ **Limitations to be aware of**
- Queries can hallucinate structure
- Risk increases with larger schemas
- Ambiguous questions may produce unsafe queries

---

### 🏗️ Production Approach: Graph as a Tool

In production systems, graph access is typically exposed through **validated tools or APIs**.

🛡️ **What this enables**
- Input validation and constraints
- Predefined, tested query logic
- Structured and predictable outputs

🧠 **Shift in the LLM’s role**
- The LLM decides *which* tool to use
- The tool controls *how* the query is executed

---

### ⭐ Key Takeaway

**Prototypes favor flexibility.  
Production systems favor control.**

Modern agent frameworks combine both — allowing LLMs to reason about intent while relying on tools to enforce correctness and safety.
